## 1. Import Learners
Test that all learner modules can be imported successfully.

In [2]:
import numpy as np
from dml.learners.lasso import LassoLearner
from dml.learners.elastic_net import ElasticNetLearner
from dml.learners.random_forest import RandomForestLearner

print("All imports successful!")

All imports successful!


## 2. Basic Fit & Predict Test
Verify that all three learners can fit on synthetic data and return predictions of the correct shape.

In [3]:
np.random.seed(42)
n, p = 100, 5
X = np.random.randn(n, p)
y = X @ np.ones(p) + np.random.randn(n)

learners = {
    "Lasso": LassoLearner(),
    "ElasticNet": ElasticNetLearner(),
    "RandomForest": RandomForestLearner()
}

for name, learner in learners.items():
    learner.fit(X, y)
    y_pred = learner.predict(X)
    print(f"{name}: output shape = {y_pred.shape}, first 3 predictions = {y_pred[:3].round(2)}")

Lasso: output shape = (100,), first 3 predictions = [ 2.12  1.96 -4.35]
ElasticNet: output shape = (100,), first 3 predictions = [ 2.12  1.97 -4.36]
RandomForest: output shape = (100,), first 3 predictions = [ 2.32  2.96 -4.64]


## 3. CausalForest Learner Test
Verify that CausalForestLearner can be imported and produces correct output shape.

In [4]:
from dml.learners.causal_forest import CausalForestLearner

cf_learner = CausalForestLearner()
cf_learner.fit(X, y)
y_pred = cf_learner.predict(X)
print(f"CausalForest: output shape = {y_pred.shape}, first 3 predictions = {y_pred[:3].round(2)}")

CausalForest: output shape = (100,), first 3 predictions = [ 0.13  0.87 -1.05]


## 4. Cross-fitting Test
Verify that cross-fitting produces out-of-sample predictions for all observations.

In [5]:
from dml.utils.cross_fitting import cross_fit

learner = LassoLearner()
y_pred_cf = cross_fit(learner, X, y, n_splits=5, random_state=66)

print(f"Input shape: {y.shape}")
print(f"Output shape: {y_pred_cf.shape}")
print(f"Residuals (first 5): {(y - y_pred_cf)[:5].round(3)}")

Input shape: (100,)
Output shape: (100,)
Residuals (first 5): [ 0.991  2.292 -1.494  0.698 -0.425]


## 5. PLR Verification
Verify that PLR correctly estimates theta on synthetic data where true theta_0 = 1.

In [6]:
from dml.models.plr import PLR

# 造合成数据，真实 theta_0 = 1
np.random.seed(66)
n, p = 500, 20
X = np.random.randn(n, p)
theta_0 = 1.0
D = X @ np.random.randn(p) + np.random.randn(n)
Y = theta_0 * D + X @ np.random.randn(p) + np.random.randn(n)

# 用 Lasso 跑 PLR
plr = PLR(learner=LassoLearner(), n_splits=5, random_state=66)
plr.fit(Y, D, X)
results = plr.predict()

print(f"True theta_0:  {theta_0}")
print(f"Estimated theta: {results['theta']:.4f}")
print(f"95% CI: ({results['ci_lower']:.4f}, {results['ci_upper']:.4f})")
print(f"CI contains true theta: {results['ci_lower'] < theta_0 < results['ci_upper']}")

True theta_0:  1.0
Estimated theta: 0.9545
95% CI: (0.8652, 1.0438)
CI contains true theta: True


## 6. IRM Verification
Verify that IRM correctly estimates ATE on synthetic binary treatment data.

In [7]:
from dml.models.irm import IRM

# 造合成数据，binary treatment，真实 ATE = 1
np.random.seed(66)
n, p = 500, 20
X = np.random.randn(n, p)
theta_0 = 1.0

# propensity score
prop = 1 / (1 + np.exp(-X[:, 0]))
D = np.random.binomial(1, prop, n)

# outcome
Y = theta_0 * D + X @ np.random.randn(p) + np.random.randn(n)

# 用 Lasso 跑 IRM
irm = IRM(learner=LassoLearner(), n_splits=5, random_state=66)
irm.fit(Y, D, X)
results = irm.predict()

print(f"True ATE:        {theta_0}")
print(f"Estimated theta: {results['theta']:.4f}")
print(f"95% CI: ({results['ci_lower']:.4f}, {results['ci_upper']:.4f})")
print(f"CI contains true theta: {results['ci_lower'] < theta_0 < results['ci_upper']}")

True ATE:        1.0
Estimated theta: 0.8281
95% CI: (0.6204, 1.0358)
CI contains true theta: True


## 7. Clean Import Test
Verify that the package-level imports work correctly.

In [8]:
from dml import LassoLearner, ElasticNetLearner, RandomForestLearner, CausalForestLearner
from dml import PLR, IRM

print("Package-level imports successful!")

Package-level imports successful!


## 8. Neural Network Learner Test
Verify that NeuralNetLearner works as a drop-in replacement for other learners.

In [9]:
from dml.learners.neural_net import NeuralNetLearner

np.random.seed(66)
n, p = 200, 10
X = np.random.randn(n, p)
y = X @ np.ones(p) + np.random.randn(n)

nn_learner = NeuralNetLearner(hidden_sizes=[64, 32], random_state=66)
nn_learner.fit(X, y)
y_pred = nn_learner.predict(X)

print(f"NeuralNet: output shape = {y_pred.shape}")
print(f"First 3 predictions = {y_pred[:3].round(2)}")

# 和其他 learner 对比 MSE
mse_nn = np.mean((y - y_pred) ** 2)
mse_baseline = np.mean((y - np.mean(y)) ** 2)
print(f"MSE (NN): {mse_nn:.4f}")
print(f"MSE (baseline): {mse_baseline:.4f}")
print(f"NN better than baseline: {mse_nn < mse_baseline}")

NeuralNet: output shape = (200,)
First 3 predictions = [-0.06  5.37 -1.65]
MSE (NN): 1.5636
MSE (baseline): 10.8453
NN better than baseline: True
